In [1]:
from response import get_response
import pandas as pd
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import warnings
warnings.filterwarnings('ignore')

c:\Users\LOQ\anaconda3\envs\nlp_gpu_new\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\LOQ\anaconda3\envs\nlp_gpu_new\Lib\site-packages\torch\compiler\__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


In [3]:
df = pd.read_csv(r'testing_data_for_RAG.csv')

In [8]:
df.head()

,index,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date,refined_issue
0,69,2024-01-31T05:14:27,individual,email,data_export,NaN,low,resolved,standard,It appears that i cannot log into my account; ...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0.0,very_negative,1.0,android,EU,2024-01-31 05:14:27,account_password_issue
1,1639,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"I'm having trouble, I'm unable to sign in; my ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,account_password_issue
2,1458,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"I'm having trouble, I can't sign in; password ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,account_password_issue
3,1494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,I am not able to log in; the password is incor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,account_password_issue
4,68,2024-01-31T05:14:27,individual,email,data_export,NaN,low,resolved,standard,Currently dealing with this issue: i cannot lo...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0.0,very_negative,1.0,android,EU,2024-01-31 05:14:27,account_password_issue


In [18]:
df[['initial_message','resolution_summary']].sample(2)

,initial_message,resolution_summary
61,"there is a problem, i changed my subscription but billing didn't update",NaN
28,It appears that i noticed a suspicious login on my account. every time I try,Ticket closed without further action after no response from the customer.


# This How to run RAG Pipeline 
- first you will make object from LLM from `LLM.py`
- second call the function `get_response` from `response.py`
- third just pass the query and the llm and function will do all work *_^ 

In [2]:
from LLM import get_llm
llm = get_llm()

In [5]:
query = 'i can not Play Fortnite'

In [6]:
response = get_response(query,llm)
print(response)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8565.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[I can not answer now !]


In [21]:
## Test Tine :) 
test = df.dropna(subset=['initial_message','resolution_summary'])

In [22]:
test.shape

(84, 20)

In [23]:
eval_data = [{'query':row['initial_message'],'truth':row['resolution_summary']} for _,row in test.iterrows()]

In [24]:
eval_data

[{'query': 'It appears that i cannot log into my account; the system says my account password is incorrect. after the last update',
  'truth': 'Reset account credentials and confirmed successful login.'},
 {'query': 'Currently dealing with this issue: i cannot log into my account; the system says my account password is incorrect. every time I try',
  'truth': 'Reset account credentials and confirmed successful login.'},
 {'query': 'Something is wrong — i cannot log into my account; the system says my login password is incorrect.',
  'truth': 'Reset account credentials and confirmed successful login.'},
 {'query': 'Can you help? i cannot sign in; the system says my account password is incorrect.',
  'truth': 'Reset account credentials and confirmed successful login.'},
 {'query': "I'm having trouble because i noticed a suspicious login on my account. every time I try",
  'truth': 'Ticket closed without further action after no response from the customer.'},
 {'query': 'Something is wrong

In [25]:
def evaluate_answer(query, answer, ground_truth, llm):
    eval_prompt = f"""
You are evaluating a RAG system.

Question: {query}
Ground Truth: {ground_truth}
Model Answer: {answer}

Score from 1 to 5 based on:
- correctness
- completeness
- no hallucination

Return ONLY a number (1-5).
"""

    score = llm.invoke(eval_prompt)
    return score.content.strip()

In [29]:
results = []

for sample in eval_data[:5]:
    query = sample["query"]
    gt = sample["truth"]

    answer = get_response(query,llm)

    score = evaluate_answer(query, answer, gt, llm)

    print("Query:", query)
    print("Answer:", answer)
    print("Ground Truth:", gt)
    print("Score:", score)
    print("-" * 50)

    results.append({
        "query": query,
        "answer": answer,
        "ground_truth": gt,
        "score": score
    })

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4659.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: It appears that i cannot log into my account; the system says my account password is incorrect. after the last update
Answer: The password issue may be related to the recent update. Try resetting your password using the "Forgot Password" feature on the login page. If the problem persists after resetting, contact support to verify your account status.
Ground Truth: Reset account credentials and confirmed successful login.
Score: 5
--------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1269.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Currently dealing with this issue: i cannot log into my account; the system says my account password is incorrect. every time I try
Answer: The account credentials were reset, and after the reset the user was able to log in successfully.
Ground Truth: Reset account credentials and confirmed successful login.
Score: 5
--------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7485.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Something is wrong — i cannot log into my account; the system says my login password is incorrect.
Answer: - Verify thepassword (check caps‑lock, special characters, and spelling).  
- If it’s still rejected, use the **“Forgot Password”** link to reset it, then log in with the new password.  
- If the issue continues, contact support with the exact error message.
Ground Truth: Reset account credentials and confirmed successful login.
Score: 5
--------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 567.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Can you help? i cannot sign in; the system says my account password is incorrect.
Answer: 

Use the "Forgot Password" or "Password Reset" feature on the login page to receive a password reset link via your registered email address. If this option is unavailable or you cannot access the email, contact customer support directly for account recovery assistance, providing identity verification details.
Ground Truth: Reset account credentials and confirmed successful login.
Score: 5
--------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1301.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '16', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774235880000'}, 'provider_name': None}}, 'user_id': 'user_3A558ZFOJPaITJOCFsJfIBBjeT7'}